In [46]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import joblib
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split , GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import FunctionTransformer
from preprocessing import clean_text, clean_texts

from sklearn.metrics import (
    accuracy_score ,
    precision_score,
    recall_score ,
    f1_score,
    classification_report , 
    confusion_matrix ,
    ConfusionMatrixDisplay
)

In [2]:
df = pd.read_csv("/home/pasali/Desktop/Data/AI-Engineer/Hafta-1/Data_ComplainRouter/data/complaints_train_5000.csv")
df.head()

,Product,Consumer complaint narrative,Complaint ID
0,"Credit reporting, credit repair services, or o...","In XX/XX/XXXX, my XXXX XXXX was repossessed. I...",3056081
1,Debt collection,XXXX OF XXXX REPRESENTING A PAYDAY LOAN COMPAN...,2003744
2,Debt collection,I have paid my son 's all bills at XXXX XXXX X...,1807880
3,Credit reporting,i did not allow this company to do a hard cred...,2262473
4,Credit reporting,Numerous entries on credit report for the same...,1776220


In [3]:
label_map = {
    "Credit reporting, credit repair services, or other personal consumer reports": 0,
    "Debt collection": 1,
    "Credit reporting": 2,
    "Mortgage":3 
}

df["label"] = df["Product"].map(label_map)

In [24]:
import re
def clean_text(text):

    if pd.isna(text):
        return ""

    # Küçük harfe dönüştür
    text = str(text).lower()

    # XXXX gibi gizlenmiş bilgileri kaldır
    text = re.sub(r"\bx{2,}\b", " ", text)

    # Noktalama işaretlerini kaldır
    text = re.sub(r"[^\w\s]", " ", text)

    # Fazla boşlukları temizle
    text = re.sub(r"\s+", " ", text).strip()

    return text


def clean_texts(texts):
    return [clean_text(text) for text in texts]

In [5]:
df["Consumer complaint narrative"] = df["Consumer complaint narrative"].apply(clean_text)

In [6]:
X = df["Consumer complaint narrative"]
y = df["label"]

In [7]:
X_train , X_test , y_train , y_test = train_test_split(X , y , test_size=0.2 , random_state=42 , stratify=y )

In [8]:
print("Train :" ,  X_train.shape )
print("Test :" , X_test.shape )

Train : (4000,)
Test : (1000,)


In [10]:
tf_idf_word_vectorizer  = TfidfVectorizer(max_features=3000)

X_train_tfidf = tf_idf_word_vectorizer.fit_transform(X_train)
X_test_tfidf = tf_idf_word_vectorizer.transform(X_test)


In [ ]:
model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model.fit(X_train_tfidf , y_train)

In [13]:

param_grid = {
    'penalty': ['l2'],
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'solver': ['lbfgs', 'newton-cg', 'liblinear'],
    'max_iter': [1000, 15000, 750],
}

grid = GridSearchCV(LogisticRegression(), param_grid, cv=5, scoring='accuracy')


In [15]:
grid.fit(X_train_tfidf , y_train)

/home/pasali/anaconda3/envs/myenv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/pasali/anaconda3/envs/myenv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/pasali/anacond

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",LogisticRegression()
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'C': [0.001, 0.01, ...], 'max_iter': [1000, 15000, ...], 'penalty': ['l2'], 'solver': ['lbfgs', 'newton-cg', ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose:

In [16]:
grid.best_params_

{'C': 1, 'max_iter': 1000, 'penalty': 'l2', 'solver': 'newton-cg'}

In [17]:
y_pred = grid.predict(X_test_tfidf)

In [18]:
comparison = pd.DataFrame({
    "Gerçek": y_test[:10].to_numpy(),
    "Tahmin": y_pred[:10]
})

comparison["Doğru mu?"] = comparison["Gerçek"] == comparison["Tahmin"]

print(comparison)

   Gerçek  Tahmin  Doğru mu?
0       1       1       True
1       2       2       True
2       0       1      False
3       3       3       True
4       0       0       True
5       2       2       True
6       1       1       True
7       2       0      False
8       0       1      False
9       0       2      False


In [19]:
# 1. Accuracy
accuracy = accuracy_score(y_test, y_pred)

# 2. Macro Precision
precision = precision_score(
    y_test, y_pred,
    average="macro",
    zero_division=0
)

# 3. Macro Recall
recall = recall_score(
    y_test, y_pred,
    average="macro",
    zero_division=0
)

# 4. Macro F1
f1 = f1_score(
    y_test, y_pred,
    average="macro",
    zero_division=0
)

# Sonuçlar
print(f"Accuracy:        {accuracy * 100:.2f}%")
print(f"Macro Precision: {precision * 100:.2f}%")
print(f"Macro Recall:    {recall * 100:.2f}%")
print(f"Macro F1:        {f1 * 100:.2f}%")

# 5. Classification Report
print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    zero_division=0
))

Accuracy:        72.60%
Macro Precision: 72.09%
Macro Recall:    72.60%
Macro F1:        72.31%

Classification Report:
              precision    recall  f1-score   support

           0       0.58      0.55      0.56       250
           1       0.79      0.80      0.79       250
           2       0.63      0.62      0.63       250
           3       0.88      0.94      0.91       250

    accuracy                           0.73      1000
   macro avg       0.72      0.73      0.72      1000
weighted avg       0.72      0.73      0.72      1000



Soru : 3000 özellikli veriyle eğitilmiş bir modele 1500 özellikli veri verirsek ne olur?

In [ ]:
# 1- Model sabit 3000 giriş özelliği bekliyorsa boyut uyuşmazlığı hatası oluşur. Çünkü giriş katmanının ağırlıkları 3000 özelliğe göre oluşturulmuştur.

In [48]:
cleaner = FunctionTransformer(
    func=clean_texts,
    validate=False
)

pipeline = Pipeline(
    steps=[
        ("clean_text", cleaner),

        ("tfidf", TfidfVectorizer(
            max_features=3000
        )),

        ("classifier", LogisticRegression(
            C=1,
            max_iter=1000,
            penalty="l2",
            solver="newton-cg"
        ))
    ]
)

In [49]:
pipeline.fit(X_train, y_train)

/home/pasali/anaconda3/envs/myenv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('clean_text', ...), ('tfidf', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](4,)","[0,1,2,3]"
,"func func: callable, default=NoneThe callable to use for the transformation. This will be passedthe same arguments as transform, with args and kwargs forwarded.If func is None, then func will be the identity function.",<function cle...x768fe41bd850>
,"inverse_func inverse_func: callable, default=NoneThe callable to use for the inverse transformation. This will bepassed the same arguments as inverse transform, with args andkwargs forwarded. If inverse_func is None, then inverse_funcwill be the identity function.",None
,"validate validate: bool, default=FalseIndicate that the input X array should be checked before calling``func``. The possibilities are:- If False, there is no input validation.- If True, then X will be converted to a 2-dimensional NumPy array or sparse matrix. If the conversion is not possible an exception is raised... versionchanged:: 0.22 The default of ``validate`` changed from True to False.",False
,"accept_sparse accept_sparse: bool, default=FalseIndicate that func accepts a sparse matrix as input. If validate isFalse, this has no effect. Otherwise, if accept_sparse is false,sparse matrix inputs will cause an exception to be raised.",False
,"check_inverse check_inverse: bool, default=TrueWhether to check that or ``func`` followed by ``inverse_func`` leads tothe original inputs. It can be used for a sanity check, raising awarning when the condition is not fulfilled... versionadded:: 0.20",True


In [50]:
import joblib

model_path = "/home/pasali/Desktop/Data/AI-Engineer/Hafta-1/Data_ComplainRouter/models/complaintrouter_lr.joblib"

joblib.dump(pipeline, model_path)

print("Model saved successfully.")

Model saved successfully.


In [ ]:
sample_text = "I have a problem with my mortgage payment."


joblib.dump(pipeline, "/home/pasali/Desktop/Data/AI-Engineer/Hafta-1/Data_ComplainRouter/models/complaintrouter_lr.joblib")
print("Model saved successfully.")

model = joblib.load("/home/pasali/Desktop/Data/AI-Engineer/Hafta-1/Data_ComplainRouter/models/complaintrouter_lr.joblib")
print("Model loaded successfully.")

prediction_before = pipeline.predict([sample_text])[0]

print("Prediction before saving:")
print(prediction_before)


prediction_after = model.predict([sample_text])[0]

print("Prediction after loading:", prediction_after)


print("Prediction before saving:", prediction_before)
print("Prediction after loading:", prediction_after)

if prediction_before == prediction_after:
    print("Test Passed!")
else:
    print("Test Failed!")

Model saved successfully.
Model loaded successfully.
Prediction before saving:
3
Prediction after loading: 3
Prediction before saving: 3
Prediction after loading: 3
Test Passed!


Soru : Neden joblib.load() ile internetten indirdiğimiz rastgele bir model dosyasını doğrudan açmamalıyız?

In [41]:
# 1- joblib.load(), Python'un pickle mekanizmasını kullandığı için dosya yüklenirken zararlı kod çalıştırılmasına neden olabilir. Bu yüzden yalnızca güvenilir kaynaklardan gelen modeller yüklenmeli ve dosyaların bütünlüğü kontrol edilmelidir.


In [42]:
def predict_complaint(text):

    # 1. Metnin veri tipini kontrol et
    if not isinstance(text, str):
        raise TypeError("Complaint text must be a string.")

    # 2. Metnin boş olup olmadığını kontrol et
    if not text.strip():
        raise ValueError("Complaint text cannot be empty.")

    # 3. Metni temizle
    cleaned_text = clean_text(text)

    # 4. Temizleme sonrası boşluk kontrolü
    if not cleaned_text:
        raise ValueError("Complaint text is empty after cleaning.")

    # 5. Temizlenmiş metni döndür
    return cleaned_text

In [43]:
test_texts = [
    "My mortgage payment failed!",
    "!!!",
    "    ",
    None,
    12345,
    "My BANK account is BLOCKED!!!"
]

for text in test_texts:
    try:
        result = predict_complaint(text)
        print(f"Input: {text!r}")
        print(f"Output: {result}\n")

    except (ValueError, TypeError) as e:
        print(f"Input: {text!r}")
        print(f"Error: {e}\n")

Input: 'My mortgage payment failed!'
Output: my mortgage payment failed

Input: '!!!'
Error: Complaint text is empty after cleaning.

Input: '    '
Error: Complaint text cannot be empty.

Input: None
Error: Complaint text must be a string.

Input: 12345
Error: Complaint text must be a string.

Input: 'My BANK account is BLOCKED!!!'
Output: my bank account is blocked



In [47]:
from preprocessing import clean_texts
print(clean_texts.__module__)

preprocessing


Soru : 1- predict() ile predict_proba() arasındaki fark nedir?
2- predict_proba() sonucu neden birden fazla olasılık içerir?
3- Olasılığı %90 olan bir tahminin gerçekten %90 doğrulukla yapılmış olduğunu söyleyebilir miyiz?

In [ ]:
1- """
predict(), modelin tahmin ettiği sınıf etiketini döndürür.
predict_proba() ise modelin her sınıfa ait olma olasılığını hesaplar.
 """

2- """
Çünkü modelimiz çok sınıflı sınıflandırma (Multiclass Classification) yapmaktadır.
Model, dört kategorinin her biri için ayrı bir olasılık hesaplar. Bu olasılıkların toplamı 1'dir.
En yüksek olasılığa sahip kategori tahmin sonucu olarak seçilir.

"""

3- """

Hayır. Modelin %90 olasılık üretmesi, tahminin kesin olarak %90 güvenilir olduğunu göstermez. Olasılık kalibrasyonu (Probability Calibration), modelin tahmin ettiği olasılıkların gerçek sonuçlarla ne kadar uyumlu olduğunu ölçer. İyi kalibre edilmiş bir modelde, yaklaşık %90 olasılık verilen çok sayıda tahminin yaklaşık %90'ının doğru çıkması beklenir.Model %90 olasılık verdiği tahminlerin yalnızca %60'ında doğru çıkıyorsa model aşırı güvenlidir (Overconfident).
"""